# 🧪 Batch image scaling with Fiji/ImageJ

This notebook processes Bio-Formats-compatible microscopy images one at a time:

1. 🔌 Connect to Fiji/ImageJ.
2. 📂 Open one image with Bio-Formats.
3. 📐 Scale the image using the selected factors.
4. 💾 Save the scaled image beside the original.
5. 🧹 Close all Fiji images before continuing, keeping memory usage low.

The last cell controls which images are processed and starts the batch.

## 1. 🔌 Connect to Fiji/ImageJ

Run the next code cell once to start Fiji and make it available to Python. The image-processing functions below use this connection.

In [ ]:
# Initialize the Fiji/ImageJ bridge
import os
from pathlib import Path

os.environ.setdefault("DISPLAY", ":1")

import imagej

FIJI_PATH = Path("/home/jovyan/Fiji.app")
ij = imagej.init(str(FIJI_PATH), mode="interactive")

ij.ui().showUI()
print(f"ImageJ version: {ij.getVersion()}")

## 2. 🧩 Define the image-processing steps

The next code cell defines three reusable functions:

- 📂 `read_image()` opens one file with the Bio-Formats Importer.
- 📐 `scale_image()` applies the horizontal, vertical, and z-axis scale factors.
- 💾 `save_scaled_image()` exports the result as an uncompressed OME-TIFF.

These functions do not process any files until the final batch cell is run.

In [ ]:
def read_image(image_path, range_values=None):
    """Open one image with Bio-Formats and return its ImageJ object."""
    image_path = Path(image_path).expanduser().resolve()
    if not image_path.is_file():
        raise FileNotFoundError(f"Image not found: {image_path}")

    range_values = range_values or {}
    range_args = " ".join(
        f"{name}={value}"
        for name, value in range_values.items()
        if value is not None
    )
    import_options = (
        f"open=[{image_path.as_posix()}] autoscale color_mode=Default "
        "rois_import=[ROI manager] view=Hyperstack stack_order=XYCZT "
        "use_virtual_stack"
    )
    if range_args:
        import_options = f"{import_options} specify_range {range_args}"

    ij.py.run_macro(f'''run("Bio-Formats Importer", "{import_options}");''')
    image = ij.py.active_imageplus()
    print(f"Imported: {image_path.name}")
    return image


def scale_image(image, scale_x=0.2, scale_y=None, scale_z=1.0):
    """Scale the active image and return the newly created scaled image."""
    scale_y = scale_x if scale_y is None else scale_y
    original_size = (image.getWidth(), image.getHeight(), image.getNSlices())
    scaled_size = tuple(
        round(size * factor)
        for size, factor in zip(original_size, (scale_x, scale_y, scale_z))
    )
    scale_macro = (
        f'run("Scale...", "x={scale_x} y={scale_y} z={scale_z} '
        f'width={scaled_size[0]} height={scaled_size[1]} depth={scaled_size[2]} '
        'interpolation=Bilinear average create");'
    )
    ij.py.run_macro(scale_macro)
    print(f"Scaled {original_size} to {scaled_size}")
    return ij.py.active_imageplus()


def save_scaled_image(image, output_path):
    """Save the active scaled image as an uncompressed OME-TIFF."""
    output_path = Path(output_path).expanduser().resolve()
    ij.py.run_macro(
        f'''run("Bio-Formats Exporter", "save=[{output_path.as_posix()}] '''
        '''export compression=Uncompressed");'''
    )
    print(f"Saved: {output_path}")

## 3. ⚙️ Choose images and run the batch

Edit the next cell before running it:

- 📝 Set `IMAGE_PATHS` to a list for specific files.
- 📁 Leave `IMAGE_PATHS = None` to analyze every file matching `IMAGE_PATTERN` in `IMAGE_DIRECTORY`.
- 📏 Adjust `SCALE_X`, `SCALE_Y`, and `SCALE_Z`; leaving `SCALE_Y = None` keeps the proportions in x and y.
- 🧬 Fill in `RANGE_VALUES` only when selecting specific channels or z-slices.

Each output is written beside its source image with `_rescaled.ome.tif` added to the filename. Fiji images are closed automatically after every file.

In [ ]:
# Select images and run the batch
# Option 1: provide explicit files. Set to None to use directory + pattern below.
IMAGE_PATHS = None
# IMAGE_PATHS = [
#     Path("/path/to/first_image.nd2"),
#     Path("/path/to/second_image.nd2"),
# ]

# Option 2: analyze every matching image in a directory.
IMAGE_DIRECTORY = Path.cwd()  # Usually the notebook's working directory
IMAGE_PATTERN = "*.nd2"

# Optional Bio-Formats acquisition range. Use None for each setting to keep all data.
RANGE_VALUES = {
    "c_begin": None,
    "c_end": None,
    "c_step": None,
    "z_begin": None,
    "z_end": None,
    "z_step": None,
}

SCALE_X = 0.2
SCALE_Y = None  # None means use SCALE_X
SCALE_Z = 1.0

image_paths = (
    [Path(path).expanduser() for path in IMAGE_PATHS]
    if IMAGE_PATHS is not None
    else sorted(IMAGE_DIRECTORY.expanduser().glob(IMAGE_PATTERN))
)
if not image_paths:
    raise FileNotFoundError("No images matched the selected paths or pattern.")

total_images = len(image_paths)
for image_number, input_path in enumerate(image_paths, start=1):
    input_path = input_path.resolve()
    remaining_images = total_images - image_number
    print(
        f"\n🔄 Analyzing image {image_number}/{total_images}: "
        f"{input_path.name}"
    )
    try:
        image = read_image(input_path, RANGE_VALUES)
        scaled_image = scale_image(image, SCALE_X, SCALE_Y, SCALE_Z)
        output_path = input_path.with_name(f"{input_path.stem}_rescaled.ome.tif")
        save_scaled_image(scaled_image, output_path)
    finally:
        ij.py.run_macro('run("Close All");')
        print(
            f"🧹 Closed Fiji images. Remaining images: {remaining_images}"
        )

print(f"✅ Finished processing {total_images} image(s).")